In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

colours = sns.color_palette("muted")
sns.set_theme(style='whitegrid')

/Users/aedanbullen/Desktop/Candidatures/Trading Simulation/Trading-Simulation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
#=== configuration =======
symbol = "AAPL"
start = "2015-01-01"
end = '2025-01-01'
starting_cash = 10000
n_sims = 1000 # number of Monte Carlo sims per strategy
conf_lo = 5 # lo percentile for confidence band
conf_hi = 95 # high ---------"----------------
seed = 42

In [3]:
#======== download and clean ======

raw = yf.download(symbol, start=start, end=end, auto_adjust=True)
price = raw["Close"].squeeze().dropna() # remember squeeze() gets rid of titles in series.
daily_returns = price.pct_change().fillna(0)

n_days = len(price)
print(f"{symbol}: {n_days} trading days loaded {start} to {end}")

[*********************100%***********************]  1 of 1 completed

AAPL: 2516 trading days loaded 2015-01-01 to 2025-01-01


In [4]:
# === strategies (copied from trading_stratefy_tests.ipynb) ===
buy_and_hold = pd.Series(1, index=price.index) # makes a series filled with 1s, with name dates.
#1 means to hold/buy every single day, always in the market. Most basic strategy, so used to compare other strategies.
#Effectively just never selling the stock. Watching what happens to the price.

momentum = (price.diff() > 0).astype(int) # if price goes up, then buy. If price.diff() between days is
#>0, then buy. astype(int) converts True/False to 1/0.

short_ma = price.rolling(20).mean() # buy when short-term avg > long term avg.
long_ma = price.rolling(50).mean() # rolling function makes a rolling window looking at last n days, taking average.
ma_cross = (short_ma > long_ma).astype(int) # this is the MAC method.

delta = price.diff()
gain = delta.clip(lower=0).rolling(14).mean() # if negative change, clip to 0. if positive keep.
loss = -delta.clip(upper=0).rolling(14).mean() # if positive clip to 0, if negative keep.
rsi = 100 - (100 / (1 + gain / loss)) # formula relative strength index RSI = 100 - 100 / (1 + RS)
# if RSI is near 100, mostly all gains recently, so overbought stock.
# if near 0, almost all losses recently, so oversold.
rsi_signal = (rsi < 30).astype(int) # buy when oversold.
#Remember buy low, sell high. Bets that the market overreacts when oversold, that price will snap back to true value afterwards. thus profit.
#Idea called mean reversion, idea that prices tend to drift back to long term avg.

ma = price.rolling(20).mean()
std = price.rolling(20).std()
lower_band = ma - 2 * std #lower band sits at 2std below mean (price below this is unusual/significant)
bb_signal = (price < lower_band).astype(int) # buy if drops below lower band, as it's cheap.



strategies = {
    "HOLD": buy_and_hold,
    "MOM": momentum,
    "Moving Average Crossover": ma_cross,
    "RSI": rsi_signal,
    "Bollinger Bands": bb_signal

}

In [ ]:
# ====== Helper functions =====

def equity_curve(signal: pd.Series, returns: pd.Series) -> pd.Series:
    shifted_signal = signal.shift(1).fillna(0) # 1 day later. Cannot use today's signal to trade today.
    # the signal is the 1/0 trade/don't boolean. Also fills any nans.

    daily_growth = 1 + shifted_signal * daily_returns # shifted signal (using whatever strategy) times returns
    # what growth u get if you use said method.
    # if signal 0 (out of market) 1 + 0 * return = 1.0, no change
    # if signal 1 (in market) 1 + 1 * return = ...
    # NB 1+ because of percentage change.

    return daily_growth.cumprod() * starting_cash # compound over time. Multiplies daily growth factors and original stock.

def risk_metrics(curve: pd.Series) -> dict:
    """ compute CAGR, sharpe, max drawdown, and 5th ptl final value"""

    returns = curve.pct_change().dropna() # daily percentage changes. have dropna due to no 0th day to cf 1st day.
    trading_years = len(curve) / 252

    # compound annual growth rate.
    cagr = (curve.iloc[-1] / starting_cash) ** (1 / trading_years) - 1  # final = starting x rate ^ years. rearrange for rate.

    # find Sharpe
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0 # div by 0 guard. also set risk-free rate to 0
    # sharpe asks how much return getting per unit risk. Average daily return/volatility (std), scaled to annual by root 252. 
    # think of as a parameter to assess how much worth a risk had. 1.5 is considered strong. less than 1 is bad.

    # max drawdown.
    rolling_max = curve.cummax() # tracks portfolios all-time high over time. 
    drawdown = (curve - rolling_max) / rolling_max # how far below the peak you are (zero or negative.)
    max_drawdown = drawdown.min() # worst, most negative point.
    #MD asks what the worst peak to trough loss experienced is. 

    return {"cagr": cagr, "sharpe": sharpe, "max_drawdown": max_drawdown, "final": curve.iloc[-1]}

In [ ]:
# ====== Monte Carlo =======

rng = np.random.default_rng(seed) # rnd numner gen
ret_array = daily_returns.values # converts pd series into Numpy array just because it's easier

mc_results = {name: [] for name in strategies} # this gives a structure like {"HOLD": [], "Momentum": [], "MA Crossover": [], ...}
mc_curves = {name: [] for name in strategies}

for sim in range(n_sims):

    idx_boot = rng.integers(0, n_days, size=n_days) #draws n_days random integers between 0 and n_days. so the indices are sampled w/ replacement.
    boot_rets = pd.Series(ret_array[idx_boot], index=daily_returns.index) #pandas series of scrambled returns data.

    for name, signal in strategies.items(): # executes functions
        curve = equity_curve(signal, boot_rets)
        mc_results[name].append(curve.iloc[-1])
        mc_curves[name].append(curve.values)

print(f"Done: {n_sims} simulations {len(strategies)} strategies")

Done: 1000 simulations 5 strategies
